## Computational Physics -- Problem Set 11
##### _Theodore Landa_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["cmr10"],        # Computer Modern Roman
    "mathtext.fontset": "cm",       # Computer Modern math font
    "axes.formatter.use_mathtext": True
})
import emcee
import scipy as sp
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

_(a)_ (Exercise 1) Check out my plot! The standard uncertainty variance on the slope is $\sigma_{m}^2 = 0.0116$.

_(b)_ (Exercise 2) Check out my plot! The standard uncertainty variance on the slope is $\sigma_{m}^2 = 0.00599$. Now we're including a bunch of outliers, relative to points 5-20. However, this is certainly not captured in the standard uncertainty variance on the slope; $\sigma_{m}^2$ is much lower this time around. This is weird on the face of it, because the line is a much worse "fit" to the data here. But $\sigma_{m}^2$ has nothing to do with how well we've chosen our model (we chose linear), it just has to do with how the uncertainties at certain x-values propagate to the $\chi^2$-minimizing result. So it's certainly conceivable that we could add data that obviously screw up the linearity but absolutely make us less uncertain of the slope, given that we have chosen a linear model.

_(c)_ (Exercise 3) Check out my plot! The standard uncertainty variance on $m$ is $\sigma_{m}^2 = 0.336$.

In [ ]:
fn = "fitting_a_model_data.txt"
with open(fn):
    data = np.loadtxt(fn, delimiter=",", skiprows=0)

In [ ]:
data_1 = data[4:,1:4]
X_1 = np.ones((data_1.shape[0], 2))
X_1[:,1] = data_1[:,0]
Y_1 = data_1[:,1]
Cinv_1 = sp.sparse.diags(1 / data_1[:,2]**2)
Ca_1 = np.linalg.solve(X_1.T @ Cinv_1 @ X_1, np.eye(2))
ahat_1 = Ca_1 @ X_1.T @ Cinv_1 @ Y_1

In [ ]:
print(ahat_1)
print(Ca_1)
print(np.sqrt(Ca_1[0,0]), np.sqrt(Ca_1[1,1]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
xplot = np.arange(300)
ax.errorbar(data_1[:,0], Y_1, data_1[:,2], fmt="ok", capsize=3, label="data")
ax.plot(xplot, ahat_1[0] + ahat_1[1] * xplot, "k", label=r"$y = (2.24 \pm 0.11)\,x + (34 \pm 18)$")
ax.set_xlim(0, 300)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_ylim(0, 700)
ax.set_title("Data points 5-20 and linear fit")
ax.legend(loc="lower right")
plt.show()

In [ ]:
data_2 = data[:,1:4]
X_2 = np.ones((data_2.shape[0], 2))
X_2[:,1] = data_2[:,0]
Y_2 = data_2[:,1]
Cinv_2 = sp.sparse.diags(1 / data_2[:,2]**2)
Ca_2 = np.linalg.solve(X_2.T @ Cinv_2 @ X_2, np.eye(2))
ahat_2 = Ca_2 @ X_2.T @ Cinv_2 @ Y_2

In [ ]:
print(ahat_2)
print(Ca_2)
print(np.sqrt(Ca_2[0,0]), np.sqrt(Ca_2[1,1]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.errorbar(data_2[:,0], Y_2, data_2[:,2], fmt="ok", capsize=3, label="data")
ax.plot(xplot, ahat_2[0] + ahat_2[1] * xplot, "k", label=r"$y = (1.08 \pm 0.08)\,x + (213 \pm 14)$")
ax.set_xlim(0, 300)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_ylim(0, 700)
ax.set_title("All data and linear fit")
ax.legend(loc="lower right")
plt.show()

In [ ]:
X_3 = np.ones((data_1.shape[0], 3))
X_3[:,1] = data_1[:,0]
X_3[:,2] = data_1[:,0]**2
Ca_3 = np.linalg.solve(X_3.T @ Cinv_1 @ X_3, np.eye(3))
ahat_3 = Ca_3 @ X_3.T @ Cinv_1 @ Y_1

In [ ]:
print(ahat_3)
print(Ca_3)
print(np.sqrt(Ca_3[0,0]), np.sqrt(Ca_3[1,1]), np.sqrt(Ca_3[2,2]))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
xplot = np.arange(300)
ax.errorbar(data_1[:,0], Y_1, data_1[:,2], fmt="ok", capsize=3, label="data")
ax.plot(xplot, ahat_3[0] + ahat_3[1] * xplot + ahat_3[2] * xplot**2, "k", 
        label=r"$y = (0.0023 \pm 0.0020)\,x^2 + (1.60 \pm 0.58)\,x + (73 \pm 39)$")
ax.set_xlim(0, 300)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_ylim(0, 700)
ax.set_title("Data points 5-20 and quadratic fit")
ax.legend(loc="lower right")
plt.show()

### Problem 2.

(Exercise 5) We have $$\chi^2=[\mathbf{Y}-\mathbf{A\,X}]^T\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}]$$ where $\mathbf{Y}$ are the $y$ data, $\mathbf{A}$ are the model inputs, and $\mathbf{X}$ are the model parameters, in this case $m$ and $b$. We wish to minimize $\chi^2$ as a function of $\mathbf{X}$, so we're taking a gradient and setting it to zero: $$\nabla_\mathbf{X}\,\chi^2=0$$ Naturally, we should apply the product rule: $$\begin{aligned}\nabla_\mathbf{X}\,\chi^2 &= [\nabla_\mathbf{X}\,(\mathbf{Y}-\mathbf{A\,X})^T]\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}]+[\mathbf{Y}-\mathbf{A\,X}]^T\,\mathbf{C}^{-1}\,[\nabla_\mathbf{X}\,(\mathbf{Y}-\mathbf{A\,X})] \\ &= \mathbf{A}^T\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}]+[\mathbf{Y}-\mathbf{A\,X}]^T\,\mathbf{C}^{-1}\,\mathbf{A}\end{aligned}$$ Notice that both terms are scalars; that is, each is equal to its own transpose. Also, $\mathbf{C}^{-1}$ is symmetric, so both terms are actually tranposes of each other and are therefore equal; that is to say $$\begin{align}\mathbf{A}^T\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}] &= [\mathbf{A}^T\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}]]^T \\ &= [\mathbf{Y}-\mathbf{A\,X}]^T\,\mathbf{C}^{-1\,T}\,\mathbf{A} \\ &= [\mathbf{Y}-\mathbf{A\,X}]^T\,\mathbf{C}^{-1}\,\mathbf{A}\end{align}$$ Therefore, $$\nabla_\mathbf{X}\,\chi^2 = 2\,\mathbf{A}^T\,\mathbf{C}^{-1}\,[\mathbf{Y}-\mathbf{A\,X}]$$ Setting this to $0$, we have $$0=\mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{Y}-\mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{A\,\hat{X}}$$ $$\mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{Y} = \mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{A\,\hat{X}}$$ $$\mathbf{\hat{X}} = [\mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{A}]^{-1}\,\mathbf{A}^T\,\mathbf{C}^{-1}\,\mathbf{Y}$$

### Problem 3.

(Exercise 6) My MAP line is $y = 2.26\,x + 32.2$. I absolutely like the MAP line so much better than problem 1 b / exercise 2! It's a lot closer to problem 1 a / exercise 1, but I bet this is even more accurate.

In [ ]:
def log_prob(pars):
    b, m, P_b, Y_b, V_b = pars
    if not (0 < m < 4 and -100 < b < 400 and 0 < P_b < 1 and 100 < Y_b < 700 and 100 < V_b < 700):
        return -np.inf
    else:
        return np.sum(np.log(((1 - P_b) / np.sqrt(2 * np.pi * data_2[:,2]**2)) \
                         * np.exp(-(data_2[:,1] - m * data_2[:,0] - b)**2 / (2 * data_2[:,2]**2)) \
                         + (P_b / np.sqrt(2 * np.pi * (V_b + data_2[:,2]**2))) \
                         * np.exp(-(data_2[:,1] - Y_b)**2 / (2 * (V_b + data_2[:,2]**2))))) - np.log(V_b)

In [ ]:
ndim, nwalkers = 5, 400
p0 = np.random.rand(nwalkers, ndim) * np.array([200, 1.6, 1, 600, 600]) + np.array([-70, 1.5, 0, 100, 100])
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
state = sampler.run_mcmc(p0, 100)
sampler.reset()
sampler.run_mcmc(state, 10000);

In [ ]:
samples = sampler.get_chain(flat=True)

fig, ax = plt.subplots(figsize=(5, 5))
hist2d = ax.hist2d(samples[:,0], samples[:,1], 180, cmap="gray_r")
ax.set_xlabel("b")
ax.set_ylabel("m")
ax.set_title("MCMC-sampled marginalized posterior for the mixed model")
plt.show()

In [ ]:
freq, xe, ye, _ = hist2d
i, j = np.unravel_index(np.argmax(freq), freq.shape)
b_MAP = 0.5 * (xe[i] + xe[i+1])
m_MAP = 0.5 * (ye[j] + ye[j+1])
print(b_MAP, m_MAP)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.errorbar(data_2[:,0], Y_2, data_2[:,2], fmt="ok", capsize=3, label="data")
ax.plot(xplot, b_MAP + m_MAP * xplot, "k", label=r"MAP line: $y = 2.26\,x + 32.2$")
for i in range(10):
    ax.plot(xplot, samples[i,0] + samples[i,1] * xplot, "k", alpha=0.2, lw=0.8)
ax.plot(xplot, samples[10,0] + samples[10,1] * xplot, "k", alpha=0.2, lw=0.8, label="posterior sampled line")
ax.set_xlim(0, 300)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_ylim(0, 700)
ax.set_title("All data and MAP line with a draw from sampling")
ax.legend(loc="lower right")
plt.show()

### Problem 4.

(Exercise 8) The standard uncertainty variance on the slope (using the uncertainty tensor from Problem 1 part b / Exercise 2) is $\sigma_{m}^2 = 0.00599$. My jackknife estimate is $\sigma_{m}^2 = 0.764$ and my bootstrap estimate is $\sigma_{m}^2 = 0.340$. These all disagree! If god gave us the uncertainties on the data (or, we are very confident in our uncertainties on the data), then we can conclude from this disagreement that our model is shitty (though we already knew that from eyeball-science on the plot from problem part b / exercise 2). I think the most reasonable uncertainty is probably either the jackknife or the bootstrap estimate. I probably believe the jackknife most, given the estimated $m$ from problem 1 part a / exercise 1. That is, if I believe that my m value is going to be more accurate when I remove outliers, and if the difference between the $m$ estimator from the inliers and the whole dataset is as big as it is, then the estimated variance should be big.

In [ ]:
rng = np.random.default_rng(3)
M = 20
boot_draw_ind = rng.integers(data_2.shape[0], size=(M, data_2.shape[0]))
boot_draw = data_2[boot_draw_ind,:]
ms_boot = np.zeros(M)
for i in range(M):
    data_boot = boot_draw[i,:]
    X_boot = np.ones((data_boot.shape[0], 2))
    X_boot[:,1] = data_boot[:,0]
    Y_boot = data_boot[:,1]
    Cinv_boot = sp.sparse.diags(1 / data_boot[:,2]**2)
    Ca_boot = np.linalg.solve(X_boot.T @ Cinv_boot @ X_boot, np.eye(2))
    ms_boot[i] = (Ca_boot @ X_boot.T @ Cinv_boot @ Y_boot)[1]
sigma_m_boot = np.mean((ms_boot - ahat_2[1])**2)
print(sigma_m_boot)

In [ ]:
ms_jack = np.zeros(data_2.shape[0])
for i in range(data_2.shape[0]):
    data_jack = np.delete(data_2, i, axis=0)
    X_jack = np.ones((data_jack.shape[0], 2))
    X_jack[:,1] = data_jack[:,0]
    Y_jack = data_jack[:,1]
    Cinv_jack = sp.sparse.diags(1 / data_jack[:,2]**2)
    Ca_jack = np.linalg.solve(X_jack.T @ Cinv_jack @ X_jack, np.eye(2))
    ms_jack[i] = (Ca_jack @ X_jack.T @ Cinv_jack @ Y_jack)[1]
sigma_m_jack = np.mean((ms_jack - np.mean(ms_jack))**2) * (data_2.shape[0] - 1)
print(sigma_m_jack)

### Credits.

Credit to the Numpy, Scipy, and emcee documentation, and to David Hogg, Jo Bovy, and Dustin Lang for coauthoring this chapter of Data analysis recipes. Also credit to Claude for showing me how to properly set up priors, and for helping me find np.unravel_index.